# AI Repair Agent

## Objective

This notebook analyzes quarantined records generated by the Silver layer using a Large Language Model (LLM).

Responsibilities:

- Read quarantined datasets
- Analyze unresolved anomalies
- Generate repair recommendations
- Assign confidence scores
- Create AI-approved dataset
- Merge approved records with Silver

## Step 1 - Import Libraries

In [0]:
import os
import json
import pandas as pd

from pyspark.sql.functions import *
from pyspark.sql.types import *

## Step 2 - Load Configuration

In [0]:
from utils.config import *

## Step 3 - Load Latest Silver Batch

In [0]:
silver_batches = sorted(os.listdir(SILVER_PATH))

LATEST_BATCH = silver_batches[-1]

print("Latest Silver Batch:")
print(LATEST_BATCH)

Latest Silver Batch:
20260806_182952


In [0]:
LATEST_SILVER_PATH = os.path.join(
    SILVER_PATH,
    LATEST_BATCH
)

print(LATEST_SILVER_PATH)

/Workspace/Users/mabd95026@gmail.com/SmartCity-AI-SelfHealing-ETL/Data/silver/20260806_182952


## Step 4 - Load Silver and Quarantine Datasets

In [0]:
bus_silver_df = spark.read.parquet(
    os.path.join(LATEST_SILVER_PATH, "bus_gps")
)

emergency_silver_df = spark.read.parquet(
    os.path.join(LATEST_SILVER_PATH, "emergency")
)

bus_quarantine_df = spark.read.parquet(
    os.path.join(LATEST_SILVER_PATH, "bus_quarantine")
)

emergency_quarantine_df = spark.read.parquet(
    os.path.join(LATEST_SILVER_PATH, "emergency_quarantine")
)

In [0]:
print("="*60)
print("Bus Silver :", bus_silver_df.count())
print("Emergency Silver :", emergency_silver_df.count())
print("Bus Quarantine :", bus_quarantine_df.count())
print("Emergency Quarantine :", emergency_quarantine_df.count())

Bus Silver : 10000
Emergency Silver : 7468
Bus Quarantine : 0
Emergency Quarantine : 532


 Install Gemini SDK

In [0]:
%pip install -q google-genai

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Initialize Gemini

In [0]:
from google import genai
from gemini_config import GEMINI_API_KEY

In [0]:
client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("✅ Gemini initialized successfully.")

✅ Gemini initialized successfully.


## Test Gemini Connection

In [0]:
response = client.models.generate_content(
    model="gemini-flash-latest",
    contents="Reply with exactly one word: READY"
)
print(response.text)

READY


## AI Prompt Template

In [0]:
def build_prompt(record):

    return f"""
You are an AI Data Quality Engineer for a Smart City platform.

Your responsibility is to infer the MOST LIKELY missing value using the available information.

Record:

Incident ID: {record["incident_id"]}
Zone: {record["zone"]}
Incident Type: {record["incident_type"]}
Severity: {record["severity"]}
Response Time: {record["response_time"]}
Status: {record["status"]}

Task:

1. Infer the most likely Zone.
2. Do NOT reply "I don't know" unless there is absolutely no reasonable inference.
3. Assign a confidence score between 0 and 100.
4. If confidence >= 90:
   action = AUTO_APPROVE
5. Otherwise:
   action = HUMAN_REVIEW

Return ONLY valid JSON.

Example:

{{
    "recommendation":"Zone B",
    "confidence":93,
    "reason":"Medical incidents with similar characteristics are commonly observed in Zone B.",
    "action":"AUTO_APPROVE"
}}
"""

## Test AI on One Record

In [0]:
sample_record = (
    emergency_quarantine_df
    .limit(1)
    .toPandas()
    .to_dict("records")[0]
)
sample_record

{'incident_id': 'INC10036',
 'zone': 'Unknown',
 'incident_type': 'Medical',
 'severity': 4,
 'response_time': 14,
 'status': 'In Progress',
 'timestamp': Timestamp('2026-08-06 08:23:16'),
 'batch_id': '20260806_082316',
 'ingestion_timestamp': Timestamp('2026-08-06 08:23:46.132736'),
 'source_system': 'emergency_simulator',
 'repair_status': 'quarantined',
 'repair_reason': 'Missing Zone',
 'repair_confidence': 0,
 'quarantine_reason': 'Unable to repair confidently',
 'quarantine_timestamp': Timestamp('2026-08-06 18:30:02.679156')}

## Analyze One Record with Gemini

In [0]:
prompt = build_prompt(sample_record)

response = client.models.generate_content(
    model="gemini-flash-latest",
    contents=prompt
)

print(response.text)

{
    "recommendation": "Zone A",
    "confidence": 65,
    "reason": "High-severity medical incidents with response times around 14 minutes frequently correlate with high-density residential or central commercial sectors such as Zone A, but spatial telemetry is required for certainty.",
    "action": "HUMAN_REVIEW"
}


## Parse AI Response

In [0]:
import json

ai_result = json.loads(response.text)

ai_result

{'recommendation': 'HUMAN_REVIEW',
 'confidence': 0,
 'reason': "The record was quarantined because the 'Zone' field is set to 'Unknown' ('Missing Zone'). The record lacks spatial attributes (such as GPS coordinates, sensor IDs, or street addresses) required to infer the correct location automatically.",
 'action': 'HUMAN_REVIEW'}

## AI Decision Summary

In [0]:
print("=" * 60)
print("AI DECISION")
print("=" * 60)

print("Recommendation :", ai_result["recommendation"])
print("Confidence     :", ai_result["confidence"])
print("Action         :", ai_result["action"])
print("Reason         :", ai_result["reason"])

AI DECISION
Recommendation : HUMAN_REVIEW
Confidence     : 0
Action         : HUMAN_REVIEW
Reason         : The record was quarantined because the 'Zone' field is set to 'Unknown' ('Missing Zone'). The record lacks spatial attributes (such as GPS coordinates, sensor IDs, or street addresses) required to infer the correct location automatically.


## AI Approval Engine

In [0]:
if ai_result["action"] == "AUTO_APPROVE":
    print("✅ Record Approved by AI")
else:
    print("⚠️ Record Sent for Human Review")

⚠️ Record Sent for Human Review


## Create AI Audit Record

In [0]:
ai_audit = {
    "incident_id": sample_record["incident_id"],
    "recommendation": ai_result["recommendation"],
    "confidence": ai_result["confidence"],
    "action": ai_result["action"],
    "reason": ai_result["reason"]
}

ai_audit

{'incident_id': 'INC10036',
 'recommendation': 'HUMAN_REVIEW',
 'confidence': 0,
 'action': 'HUMAN_REVIEW',
 'reason': "The record was quarantined because the 'Zone' field is set to 'Unknown' ('Missing Zone'). The record lacks spatial attributes (such as GPS coordinates, sensor IDs, or street addresses) required to infer the correct location automatically."}

## AI Batch Processing

In [0]:
from datetime import datetime
import json

ai_audit_records = []

quarantine_records = quarantine_records = (
    emergency_quarantine_df
    .limit(10)
    .toPandas()
    .to_dict("records")
)

print("Total Quarantine Records :", len(quarantine_records))
#emergency_quarantine_df.toPandas().to_dict("records") this is for all records but due to gemini limitation testing on 10

print("Total Quarantine Records :", len(quarantine_records))

Total Quarantine Records : 10
Total Quarantine Records : 10


In [0]:
for record in quarantine_records:

    try:

        prompt = build_prompt(record)

        response = client.models.generate_content(
            model="gemini-flash-latest",
            contents=prompt
        )

        result = json.loads(response.text)

        ai_audit_records.append({

            "incident_id": record["incident_id"],

            "recommendation": result["recommendation"],

            "confidence": result["confidence"],

            "action": (
                "AUTO_APPROVE"
                if result["confidence"] >= 60
                else "HUMAN_REVIEW"
            ),

            "reason": result["reason"],

            "processed_at": datetime.now(),

            "model": "gemini-flash-latest"

        })

    except Exception as e:

        ai_audit_records.append({

            "incident_id": record["incident_id"],

            "recommendation": None,

            "confidence": 0,

            "action": "ERROR",

            "reason": str(e),

            "processed_at": datetime.now(),

            "model": "gemini-flash-latest"

        })

In [0]:
ai_audit_records = []

In [0]:
print("AI Records Processed :", len(ai_audit_records))

AI Records Processed : 10


In [0]:
ai_audit_df = pd.DataFrame(ai_audit_records)

ai_audit_df.head()

,incident_id,recommendation,confidence,action,reason,processed_at,model
0,INC10036,None,0,ERROR,429 RESOURCE_EXHAUSTED. {'error': {'code': 429...,2026-08-07 10:59:56.791585,gemini-flash-latest
1,INC10042,None,0,ERROR,429 RESOURCE_EXHAUSTED. {'error': {'code': 429...,2026-08-07 10:59:56.874943,gemini-flash-latest
2,INC10614,None,0,ERROR,429 RESOURCE_EXHAUSTED. {'error': {'code': 429...,2026-08-07 10:59:56.964600,gemini-flash-latest
3,INC10735,None,0,ERROR,429 RESOURCE_EXHAUSTED. {'error': {'code': 429...,2026-08-07 10:59:57.039784,gemini-flash-latest
4,INC10813,None,0,ERROR,429 RESOURCE_EXHAUSTED. {'error': {'code': 429...,2026-08-07 10:59:57.133231,gemini-flash-latest


In [0]:
ai_audit_records[0]

{'incident_id': 'INC10036',
 'recommendation': 'HUMAN_REVIEW',
 'confidence': 0,
 'action': 'HUMAN_REVIEW',
 'reason': "The record was quarantined because the 'Zone' field is marked as 'Unknown'. There is insufficient spatial data (e.g., GPS coordinates, street address, or unit ID) within the record to accurately infer the missing Zone.",
 'processed_at': datetime.datetime(2026, 8, 7, 10, 42, 47, 675847),
 'model': 'gemini-flash-latest'}

In [0]:
print("=" * 60)
print("AI Decision Summary")
print("=" * 60)

print("AI Approved :", len(ai_audit_df[ai_audit_df["action"] == "AUTO_APPROVE"]))
print("Human Review :", len(ai_audit_df[ai_audit_df["action"] == "HUMAN_REVIEW"]))

AI Decision Summary
AI Approved : 0
Human Review : 0


## Create AI Audit DataFrame

In [0]:
ai_audit_df = pd.DataFrame(ai_audit_records)

ai_audit_df.head()

,incident_id,recommendation,confidence,action,reason,processed_at,model
0,INC10036,HUMAN_REVIEW,0,HUMAN_REVIEW,The record was quarantined because the 'Zone' ...,2026-08-07 10:42:47.675847,gemini-flash-latest
1,INC10042,HUMAN_REVIEW,0,HUMAN_REVIEW,The record was quarantined because the 'Zone' ...,2026-08-07 10:42:54.371034,gemini-flash-latest
2,INC10614,HUMAN_REVIEW,0,HUMAN_REVIEW,The record was quarantined because the Zone is...,2026-08-07 10:42:59.425499,gemini-flash-latest
3,INC10735,Escalate record to dispatcher or operator for ...,0,HUMAN_REVIEW,The record was quarantined because the 'Zone' ...,2026-08-07 10:43:04.478873,gemini-flash-latest
4,INC10813,HUMAN_REVIEW,95,HUMAN_REVIEW,The record was quarantined because the 'Zone' ...,2026-08-07 10:43:10.465108,gemini-flash-latest


## Separate AI Decisions

In [0]:
ai_approved_df = ai_audit_df[
    ai_audit_df["action"] == "AUTO_APPROVE"
]

human_review_df = ai_audit_df[
    ai_audit_df["action"] == "HUMAN_REVIEW"
]

In [0]:
print("=" * 60)
print("AI Decision Summary")
print("=" * 60)

print("AI Approved :", len(ai_approved_df))
print("Human Review :", len(human_review_df))

AI Decision Summary
AI Approved : 0
Human Review : 10
